## tách file train trộn data generative và gốc và test 100% gốc

In [1]:
import pandas as pd
import numpy as np

# 1. Load dữ liệu
# Giả sử file v6 tên là 'Agri_Data_SMOTE_Noise_v6.csv'
df_orig = pd.read_csv('Agri_Data_Cleaned.csv')
df_v6 = pd.read_csv('Agri_Data_SMOTE_Noise_v6.csv')

print(f"Tổng số dòng data gốc: {len(df_orig)}")

# 2. Chiến thuật lấy 179 dòng từ data gốc (Stratified Sampling)
# Mục tiêu: Phải có mặt đủ mọi District và Crop Name
target_orig_count = 179
selected_indices = set()

# Bước 1: Lấy mỗi District ít nhất 1 mẫu
groups_district = df_orig.groupby('District')
for name, group in groups_district:
    # Lấy mẫu ngẫu nhiên trong nhóm
    idx = np.random.choice(group.index, 1)[0]
    selected_indices.add(idx)

# Bước 2: Lấy mỗi Crop Name ít nhất 1 mẫu (nếu chưa có trong tập đã chọn)
# Kiểm tra xem crop nào chưa được chọn
current_crops = df_orig.loc[list(selected_indices), 'Crop Name'].unique()
missing_crops = set(df_orig['Crop Name'].unique()) - set(current_crops)

for crop in missing_crops:
    # Lấy mẫu thuộc crop này
    candidates = df_orig[df_orig['Crop Name'] == crop].index
    # Tránh lấy trùng nếu có thể (dù set đã lo việc này, nhưng để chắc chắn)
    new_candidates = list(set(candidates) - selected_indices)
    if new_candidates:
        idx = np.random.choice(new_candidates, 1)[0]
        selected_indices.add(idx)

# Bước 3: Điền cho đủ 179 dòng (nếu còn thiếu)
current_count = len(selected_indices)
if current_count < target_orig_count:
    needed = target_orig_count - current_count
    remaining_pool = list(set(df_orig.index) - selected_indices)
    
    # Lấy thêm ngẫu nhiên
    extra_indices = np.random.choice(remaining_pool, needed, replace=False)
    selected_indices.update(extra_indices)

# Chuyển thành list để truy xuất
final_orig_indices = list(selected_indices)

# Tạo dataframe con
df_train_orig_part = df_orig.loc[final_orig_indices]
df_test_final = df_orig.drop(index=final_orig_indices)

print(f"Đã chọn được {len(df_train_orig_part)} dòng từ data gốc.")
print(f"Số dòng còn lại làm tập Test: {len(df_test_final)}")

# 3. Lấy 15821 dòng từ v6
target_v6_count = 15821
if len(df_v6) >= target_v6_count:
    df_train_v6_part = df_v6.sample(n=target_v6_count, random_state=42)
else:
    print(f"Cảnh báo: v6 chỉ có {len(df_v6)} dòng, lấy toàn bộ.")
    df_train_v6_part = df_v6.copy()

# 4. Ghép lại thành tập Train hoàn chỉnh
df_train_final = pd.concat([df_train_orig_part, df_train_v6_part], ignore_index=True)

# Shuffle (trộn đều) tập train để model học tốt hơn
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Kích thước tập Train cuối cùng: {df_train_final.shape}")

# 5. Lưu file
df_train_final.to_csv('Agri_Train_Combined_16k.csv', index=False)
df_test_final.to_csv('Agri_Test_Original_4k.csv', index=False)

print("Đã xuất file: Agri_Train_Combined_16k.csv và Agri_Test_Original_4k.csv")

Tổng số dòng data gốc: 4178
Đã chọn được 179 dòng từ data gốc.
Số dòng còn lại làm tập Test: 3999
Kích thước tập Train cuối cùng: (16000, 51)
Đã xuất file: Agri_Train_Combined_16k.csv và Agri_Test_Original_4k.csv


### Xóa các cột đa tương quan

In [2]:
import pandas as pd
import os

# 1. Danh sách các file cần xử lý
input_files = ['Agri_Test_Original_4k.csv', 'Agri_Train_Combined_16k.csv']

# 2. Danh sách các cột muốn xóa
columns_to_drop = ['Rain_Temp_Ratio', 'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max', 'Is_Extreme_Heat', 'Production', 'Sand', 'CN_Ratio', 'Moisture_Ratio']

# Duyệt qua từng file trong danh sách
for filename in input_files:
    try:
        # Đọc dữ liệu từ file CSV
        df = pd.read_csv(filename)
        
        # Lọc ra những cột có tồn tại trong dữ liệu để xóa (tránh lỗi nếu 1 cột không tồn tại)
        existing_cols = [col for col in columns_to_drop if col in df.columns]

        if len(existing_cols) > 0:
            # 3. Thực hiện xóa cột
            # inplace=True để áp dụng thay đổi trực tiếp lên bảng dữ liệu hiện tại
            df.drop(columns=existing_cols, inplace=True)
            
            # 4. Tạo tên file đầu ra tự động (thêm _clean_column vào tên gốc)
            # Ví dụ: Agri_Test_Original_4k.csv -> Agri_Test_Original_4k_clean_column.csv
            base_name, ext = os.path.splitext(filename)
            output_filename = f"{base_name}_clean_column{ext}"
            
            # Lưu lại dữ liệu sau khi xóa cột ra file mới
            df.to_csv(output_filename, index=False)
            print(f"Xử lý file '{filename}': Đã xóa các cột {existing_cols} và lưu vào '{output_filename}'.")
        else:
            print(f"File '{filename}': Không tìm thấy bất kỳ cột nào trong danh sách cần xóa.")
            
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{filename}'. Kiểm tra lại tên file hoặc đường dẫn.")
    except Exception as e:
        print(f"Đã xảy ra lỗi khi xử lý file '{filename}': {e}")

Xử lý file 'Agri_Test_Original_4k.csv': Đã xóa các cột ['Rain_Temp_Ratio', 'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max', 'Is_Extreme_Heat', 'Production', 'Sand', 'CN_Ratio', 'Moisture_Ratio'] và lưu vào 'Agri_Test_Original_4k_clean_column.csv'.
Xử lý file 'Agri_Train_Combined_16k.csv': Đã xóa các cột ['Rain_Temp_Ratio', 'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max', 'Is_Extreme_Heat', 'Production', 'Sand', 'CN_Ratio', 'Moisture_Ratio'] và lưu vào 'Agri_Train_Combined_16k_clean_column.csv'.


### Chuẩn hóa thang đo (Standardization / Z-Score Scaling)

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os

# 1. Danh sách các file cần xử lý
input_files = ['Agri_Test_Original_4k_clean_column.csv', 'Agri_Train_Combined_16k_clean_column.csv']

for filename in input_files:
    try:
        # Đọc dữ liệu từ file
        df = pd.read_csv(filename)

        # 2. Lọc ra các cột là dữ liệu số (int, float)
        # Lưu ý: Lệnh này sẽ chọn tất cả các cột số, bao gồm cả biến mục tiêu (ví dụ: Yield) nếu nó là số.
        num_cols = df.select_dtypes(include=['float64', 'int64']).columns

        if len(num_cols) > 0:
            # 3. Khởi tạo thuật toán chuẩn hóa Z-score
            scaler = StandardScaler()

            # 4. Tạo bản sao và thực hiện chuẩn hóa
            # fit_transform tính toán mean và std cho từng cột rồi áp dụng chuẩn hóa
            df_scaled = df.copy()
            df_scaled[num_cols] = scaler.fit_transform(df[num_cols])

            # Tạo tên file đầu ra tự động
            # Ví dụ: Agri_Test_Original_4k_clean_column.csv -> Agri_Test_Original_4k_clean_column_scaled.csv
            base_name, ext = os.path.splitext(filename)
            output_filename = f"{base_name}_scaled{ext}"

            # 5. Lưu ra file mới
            df_scaled.to_csv(output_filename, index=False)
            print(f"File '{filename}': Đã chuẩn hóa {len(num_cols)} cột số thành công và lưu vào '{output_filename}'.")
        else:
            print(f"File '{filename}': Không tìm thấy cột dữ liệu số nào để chuẩn hóa.")

    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{filename}'.")
    except Exception as e:
        print(f"Đã xảy ra lỗi khi xử lý file '{filename}': {e}")

File 'Agri_Test_Original_4k_clean_column.csv': Đã chuẩn hóa 33 cột số thành công và lưu vào 'Agri_Test_Original_4k_clean_column_scaled.csv'.
File 'Agri_Train_Combined_16k_clean_column.csv': Đã chuẩn hóa 33 cột số thành công và lưu vào 'Agri_Train_Combined_16k_clean_column_scaled.csv'.
